# Imaging a Flare from Sagittarius A*
<hr style="border: 2px solid #f5bf03" />

- **Description:** Creating images of a flare from Sagittarius A*.
- **Level:** Intermediate
- **Data:** XMM observation of Sag A* (obsid=0112972101)
- **Requirements:** Must be run using pySAS version 2.3.0 or higher.
- **Credit:** Ryan Tanner (January 2026)
- **Support:** <a href="https://heasarc.gsfc.nasa.gov/docs/xmm/xmm_helpdesk.html">XMM Newton GOF Helpdesk</a>
- **Last verified to run:** 28 Janurary 2026, for SAS v22.1 and pySAS v2.3.0

<hr style="border: 2px solid #f5bf03" />

## 1. Introduction

This tutorial is based on Goldwurm et al. 2003, ApJ, 584, 751 (DOI [10.1086/345749](https://doi.org/10.1086/345749)).

The general process is as follows:

1. Download the data.
2. Take a quick look at the data.
3. Apply basic filters to the `MOS 1` and `MOS 2` event lists.
4. Merge the `MOS 1` and `MOS 2` event lists.
5. Select a 10 arcsecond region from the merged event list around Sag A*.
6. Select a 30 arcsecond "background" region from the merged event list for comparison.
7. Plot the light curve for the region around Sag A*. Note the flare during the last 1000 seconds of the observation. Compare to the "background" region.
8. Filter on "time" the **merged** event list into two event lists, one for 1000 seconds before the flare, and the other for the 1000 seconds during the flare.
9. Plot the events lists pre-flare and during the flare. Compare.

#### Useful Links

- [`pysas` Documentation](https://xmm-tools.cosmos.esa.int/external/sas/current/doc/pysas/index.html "pysas Documentation")
- [`pysas` on GitHub](https://github.com/XMMGOF/pysas)
- [Common SAS Threads](https://www.cosmos.esa.int/web/xmm-newton/sas-threads "SAS Threads")
- [Users' Guide to the XMM-Newton Science Analysis System (SAS)](https://xmm-tools.cosmos.esa.int/external/xmm_user_support/documentation/sas_usg/USG/SASUSG.html "Users' Guide")
- [The XMM-Newton ABC Guide](https://heasarc.gsfc.nasa.gov/docs/xmm/abc/ "ABC Guide")
- [XMM Newton GOF Helpdesk](https://heasarc.gsfc.nasa.gov/docs/xmm/xmm_helpdesk.html "Helpdesk") - Link to form to contact the GOF Helpdesk.

<div class="alert alert-block alert-warning">
    <b>Warning:</b> By default this notebook will place observation data files in your default <tt>data_dir</tt> directory. Make sure pySAS has been configured properly.
</div>

In [ ]:
# pySAS imports
import pysas
from pysas import MyTask

# Useful imports
import os

# Imports for plotting
import matplotlib.pyplot as plt
from astropy.visualization import astropy_mpl_style
from astropy.io import fits
from astropy.wcs import WCS
from astropy.table import Table
from regions import CircleSkyRegion
from astropy.coordinates import SkyCoord
import astropy.units as u
plt.style.use(astropy_mpl_style)

# To handle certain warnings
import warnings
warnings.filterwarnings("ignore")

These are some functions we will be using.

In [ ]:
def plot_zoom_in(image_file, zoom=4, x=None, y=None, vmin=1.0, vmax=10.0):
    # Open file
    hdu = fits.open(image_file)[0]
    wcs = WCS(hdu.header)
    im_shape = hdu.shape
    if x is None:
        x_center = int(im_shape[0]/2)
    else:
        x_center = x
    if y is None:
        y_center = int(im_shape[1]/2)
    else:
        y_center = y
    
    # Define the zoomed-in region
    xmin, xmax = x_center-int(x_center/(2*zoom)), x_center+int(x_center/(2*zoom))
    ymin, ymax = y_center-int(y_center/(2*zoom)), y_center+int(y_center/(2*zoom))

    print(f'Image center: ({x_center}, {y_center})')
    print(f'x limits: ({xmin}, {xmax})')
    print(f'y limits: ({ymin}, {ymax})')

    # Plot
    ax = plt.subplot(projection=wcs)
    plt.imshow(hdu.data, origin='lower', norm='log', vmin=vmin, vmax=vmax)
    ax.set_facecolor("black")
    ax.set(xlim=(xmin, xmax), ylim=(ymin, ymax))
    plt.grid(color='blue', ls='solid')
    plt.xlabel('RA')
    plt.ylabel('Dec')
    plt.colorbar()
    plt.show()

    return

def make_hires_image(in_event_list,
                     pi_min = 200,
                     pi_max = 13000,
                     ximagebinsize = 40,
                     yimagebinsize = 40,
                     out_image='image.fits',
                     output   = True):

    # Filter expression
    expression = '(PI in [{pi_min}:{pi_max}])'.format(pi_min=pi_min,pi_max=pi_max)

    inargs = {'table'         : in_event_list+':EVENTS', 
              'withimageset'  : 'yes',
              "expression"    : expression, 
              'imageset'      : out_image,
              'imagebinning'  : 'binSize',
              'xcolumn'       : 'X',
              'ycolumn'       : 'Y',
              'ximagebinsize' : ximagebinsize,
              'yimagebinsize' : yimagebinsize}
    
    MyTask('evselect', inargs, output_to_terminal = output).run()

    return

def plot_region(image_file, ra, dec, radius, vmin=1.0, vmax=10.0, zoom=1.0):
    
    # Define region
    center = SkyCoord(ra, dec)
    region = CircleSkyRegion(center, radius)
    
    # Open file
    hdu = fits.open(image_file)[0]
    wcs = WCS(hdu.header)

    # Convert region to artist object
    pixel_region = region.to_pixel(wcs)
    artist = pixel_region.as_artist(color='lime')

    # Set image limits
    # This sets the bounds of the lower left (ll) and upper right (ur) of the plot.
    # NOTE: The calculation for the ll and ur of RA is reversed from the
    # calculation for the ll and ur of the DEC (+,- vs. -,+).
    # This preserves the correct orientation of the image.
    ra_ll  = ra+100*radius/zoom
    ra_ur  = ra-100*radius/zoom
    dec_ll = dec-100*radius/zoom
    dec_ur = dec+100*radius/zoom
    ra_lim  = [ra_ll.value, ra_ur.value]
    dec_lim = [dec_ll.value, dec_ur.value]
    # The third value "0" sets the "origin", or the index of the first pixel value.
    # It is "0" because Python starts counting at "0".
    (xmin, xmax), (ymin, ymax) = wcs.all_world2pix(ra_lim, dec_lim, 0)

    # Plot
    ax = plt.subplot(projection=wcs)
    plt.imshow(hdu.data, origin='lower', norm='log', vmin=vmin, vmax=vmax)
    ax.set_facecolor("black")
    ax.add_artist(artist)
    ax.set(xlim=(xmin, xmax), ylim=(ymin, ymax))
    plt.grid(color='blue', ls='solid')
    plt.xlabel('RA')
    plt.ylabel('Dec')
    plt.colorbar()
    plt.show()

    return

def plot_images(left_image,right_image,titles,zoom=4,x=None,y=None,vmin=1.0,vmax=10.0,grid=True):
    """
    Takes file names for two FITS images and a list containing
    the titles for the two plots.

    Plots the images in a row. All images must have the same WCS.
    """
    hdu = []
    hdu.append(fits.open(left_image)[0])
    hdu.append(fits.open(right_image)[0])

    # Open file
    im_shape = hdu[0].shape
    if x is None:
        x_center = int(im_shape[0]/2)
    else:
        x_center = x
    if y is None:
        y_center = int(im_shape[1]/2)
    else:
        y_center = y
    
    # Define the zoomed-in region
    xmin, xmax = x_center-int(x_center/(2*zoom)), x_center+int(x_center/(2*zoom))
    ymin, ymax = y_center-int(y_center/(2*zoom)), y_center+int(y_center/(2*zoom))
    
    fig, axes = plt.subplots(1, 2,subplot_kw={'projection': WCS(hdu[0].header)}, figsize=(9, 9))
    plt.grid(False)
    
    for i, ax in enumerate(axes.flat):
        ax.set_facecolor("black")
        ax.grid(False)
        ax.imshow(hdu[i].data, origin='lower', norm='log', vmin=vmin, vmax=vmax)
        ax.set(xlim=(xmin, xmax), ylim=(ymin, ymax))
        ax.title.set_text(titles[i])
        ax.coords[0].set_ticklabel_visible(False)
        ax.coords[1].set_ticklabel_visible(False)
    
    plt.subplots_adjust(wspace=0)

## 2. Download PPS Files

In [ ]:
obsid = '0112972101'
my_pps = pysas.PPSFiles(obsid)
my_pps.download_PPS_data(repo='heasarc')
os.chdir(my_pps.work_dir)

Filenames we will use.

In [ ]:
mos1_filtered        = 'mos1_filtered_event_list.fits'
mos2_filtered        = 'mos2_filtered_event_list.fits'
merged_mos           = 'merged_mos_event_list.fits'
merged_image         = 'merged_mos_image.fits'
source_event_list    = 'mos_source_event_list.fits'
source_image         = 'mos_source_image.fits'
time_slice_source    = 'time_slice_source_event_list.fits'
bkg_event_list       = 'mos_bkg_event_list.fits'
bkg_image            = 'mos_bkg_image.fits'
time_slice_bkg       = 'time_slice_bkg_event_list.fits'
source_light_curve   = 'source_light_curve.fits'
bkg_light_curve      = 'bkg_light_curve.fits'
pre_flare_event_list = 'pre_flare_event_list.fits'
pre_flare_image      = 'pre_flare_image.fits'
flare_event_list     = 'flare_event_list.fits'
flare_image          = 'flare_image.fits'

In [ ]:
print('EPIC Event Lists: my_pps.EPIC_event_lists\n')
for file in my_pps.EPIC_event_lists:
    print(f' > {file}')
print('\nEPIC FITS Image Files: my_pps.EPIC_images\n')
for file in my_pps.EPIC_images:
    print(f' > {file}')

## 3. Quick Look at the Data

In [ ]:
for evtli in my_pps.EPIC_event_lists:
    my_pps.quick_eplot(evtli,vmin=1.0,vmax=100.0)

In [ ]:
for evtli in my_pps.EPIC_event_lists:
    my_pps.quick_lcplot(evtli)

## 4. Focus on Sag A*

In [ ]:
plot_zoom_in(my_pps.EPIC_images[5],vmax=100, x=338, y=345)

Perform some basic filtering.

In [ ]:
inargs = {'table'           : my_pps.EPIC_event_lists[0],
          'energycolumn'    : 'PI',
          'withfilteredset' : 'yes',
          'filteredset'     : mos1_filtered,
          'keepfilteroutput': 'yes',
          'filtertype'      : 'expression',
          'expression'      : "'(PATTERN <= 12)&&(PI in [2000:10000])&&#XMMEA_EM'"}

MyTask('evselect', inargs).run()

inargs = {'table'           : my_pps.EPIC_event_lists[1],
          'energycolumn'    : 'PI',
          'withfilteredset' : 'yes',
          'filteredset'     : mos2_filtered,
          'keepfilteroutput': 'yes',
          'filtertype'      : 'expression',
          'expression'      : "'(PATTERN <= 12)&&(PI in [2000:10000])&&#XMMEA_EM'"}

MyTask('evselect', inargs).run()

Merge the filtered event lists for `MOS 1` and `MOS 2`.

In [ ]:
inargs = {'set1'   : mos1_filtered,
          'set2'   : mos2_filtered,
          'outset' : merged_mos}

MyTask('merge', inargs).run()

Let's take a look at the merged event list.

In [ ]:
make_hires_image(merged_mos,out_image=merged_image)

In [ ]:
my_pps.quick_implot(merged_image,title='Merged MOS Image',vmin=0.1,vmax=10.0)

Let's select a region around Sag A* with a radius of 10 arcseconds.

In [ ]:
source_RA  = 266.416837 * u.deg # degrees
source_Dec = -29.007811 * u.deg # degrees
source_rad = 10.0 * u.arcsec    # arcseconds

plot_region(merged_image, source_RA, source_Dec, source_rad, vmin=1.0, vmax=100.0, zoom=5)

In [ ]:
circle = "CIRCLE({0},{1},{2})".format(source_RA.value,source_Dec.value,source_rad.to(u.deg).value)

inargs = {'table'           : merged_mos,
          'energycolumn'    : 'PI',
          'withfilteredset' : 'yes',
          'filteredset'     : source_event_list,
          'keepfilteroutput': 'yes',
          'filtertype'      : 'expression',
          'expression'      : "'((RA,DEC) in {0})'".format(circle)}

MyTask('evselect', inargs).run()

Let's take a look at the source region we selected.

In [ ]:
make_hires_image(source_event_list,out_image=source_image)

In [ ]:
plot_zoom_in(source_image,vmax=100, x=675, y=680, zoom=6)

Now let's select the last 10000 seconds of the observation ("TIME >= tslice") and look at the light curve. The start and stop times for this observation can be found in the header of the event list. We extract these times and then use them to calculate how to filter the event list using `evselect`.

In [ ]:
with fits.open(source_event_list) as hdu:
    tstart = hdu[1].header['TSTART']
    tstop  = hdu[1].header['TSTOP']

tinterval = 10000

tslice = tstop - tinterval

print(f'tstart = {tstart} \ntslice = {tslice} \ntstop  = {tstop}')

In [ ]:
inargs = {'table'           : source_event_list,
          'energycolumn'    : 'PI',
          'withfilteredset' : 'yes',
          'filteredset'     : time_slice_source,
          'keepfilteroutput': 'yes',
          'filtertype'      : 'expression',
          'expression'      : "'(TIME >= {0})'".format(tslice)}

MyTask('evselect', inargs).run()

In [ ]:
ax = my_pps.quick_lcplot(time_slice_source,light_curve_file=source_light_curve,timebinsize=180,tstart=tstop-tinterval,tend=tstop-300,title='Sag A* MOS Light Curve')

This next cell will do several things. This selects a "background" region nearby to compare the light curve of the source. It will also create a FITS file needed to plot the light curve for the background region. The FITS light curve file for the source region was created by the function `quick_lcplot` in the previous cell.

In [ ]:
bkg_RA  = (266.416837 + 0.01666666666) * u.deg # degrees
bkg_Dec = -29.007811 * u.deg # degrees
bkg_rad = 30.0 * u.arcsec    # arcseconds

plot_region(merged_image, bkg_RA, bkg_Dec, bkg_rad, vmin=1.0, vmax=10.0, zoom=5)

circle = "CIRCLE({0},{1},{2})".format(bkg_RA.value,bkg_Dec.value,bkg_rad.to(u.deg).value)

inargs = {'table'           : merged_mos,
          'energycolumn'    : 'PI',
          'withfilteredset' : 'yes',
          'filteredset'     : time_slice_bkg,
          'keepfilteroutput': 'yes',
          'filtertype'      : 'expression',
          'expression'      : "'(TIME >= {0})&&((RA,DEC) in {1})'".format(tslice,circle)}

MyTask('evselect', inargs).run()

inargs = {'table'          : time_slice_bkg, 
          'withrateset'    : 'yes',
          'rateset'        : bkg_light_curve, 
          'maketimecolumn' : 'yes', 
          'timecolumn'     : 'TIME', 
          'imagebinning'   : 'imageSize', 
          'timebinsize'    : 180, 
          'makeratecolumn' : 'yes'}

MyTask('evselect', inargs).run()

Here we plot the lightcurves for Sag A* and the background. The background has been rescaled by the factor of 0.1 for clarity. Goldwurm et al. (2003) did the same in their analysis.

In [ ]:
tstart=tstop-tinterval
tend=tstop-300
source = Table.read(source_light_curve,hdu=1)
bkg    = Table.read(bkg_light_curve,hdu=1)
plt.plot(source['TIME'],source['RATE'],bkg['TIME'],bkg['RATE']*0.1)
plt.xlabel('Time (s)')
plt.ylabel('Count Rate (ct/s)')
plt.xlim(left=tstart,right=tend)
plt.title('Sag A* and Background MOS Light Curves')
plt.show()

Compare with Goldwurm et al. (2003) Figure 1:

![Goldwurm et al. Figure 1](./_files/Goldwurm_Fig1.jpg)

## 5. Select Events Pre-Flare and During Flare

Now we return to the merged MOS event list and select the 1000 seconds before the flare, and the last 1000 seconds of the observation. We compare this to Figure 2 from Goldwurm et al. (2003).

In [ ]:
tstart_pre = tstop - 2000
tend_pre   = tstop - 1000

inargs = {'table'           : merged_mos,
          'energycolumn'    : 'PI',
          'withfilteredset' : 'yes',
          'filteredset'     : pre_flare_event_list,
          'keepfilteroutput': 'yes',
          'filtertype'      : 'expression',
          'expression'      : "'(TIME IN [{0}:{1}])'".format(tstart_pre,tend_pre)}

MyTask('evselect', inargs).run()

In [ ]:
tstart_flare = tstop - 1000

inargs = {'table'           : merged_mos,
          'energycolumn'    : 'PI',
          'withfilteredset' : 'yes',
          'filteredset'     : flare_event_list,
          'keepfilteroutput': 'yes',
          'filtertype'      : 'expression',
          'expression'      : "'(TIME >= {0})'".format(tstart_flare)}

MyTask('evselect', inargs).run()

In [ ]:
make_hires_image(pre_flare_event_list,out_image=pre_flare_image,ximagebinsize=120,yimagebinsize=120)
make_hires_image(flare_event_list,out_image=flare_image,ximagebinsize=120,yimagebinsize=120)

In [ ]:
plot_zoom_in(pre_flare_image,vmax=10, x=220, y=230, zoom=3)

In [ ]:
plot_zoom_in(flare_image,vmax=10, x=220, y=230, zoom=3)

Now we plot them side-by-side and compare with Goldwurm et al. (2003) Figure 2.

In [ ]:
plot_images(pre_flare_image,flare_image,['Pre-Flare MOS Image','Flare MOS Image'],x=220,y=230,zoom=3.5,vmin=3.0,vmax=10.0)

Compare with Goldwurm et al. (2003) Figure 2.

![Goldwurm et al. Figure 2](./_files/Goldwurm_Fig2a.jpg) ![Goldwurm et al. Figure 2](./_files/Goldwurm_Fig2b.jpg)